# Full Pipeline Step-by-Step Workflow
This notebook breaks down the `Orchestrator` to test the Distillation, Designer, and Validation loops explicitly.
It also demonstrates how downstream processes can load and utilize the new `stage_outputs` JSON structures independent of full model re-runs.


In [1]:
import sys
import os
import asyncio
import json

# Allow imports from local directory if running inside the ontology folder
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from dotenv import load_dotenv
load_dotenv()

from core.models import DocumentSource
from pipeline.orchestrator import Orchestrator


## 1. Define Test Context
Initialize the raw clinical documents that will act as the empirical context.


In [2]:
doc1 = DocumentSource(
    id="doc_001",
    text_content="""
    <ORGANIZATION>
Comprehensive Psychological Evaluation Report
Confidential – For Clinical Use Only
Patient Name: <PERSON>
Date of Birth: March 12, 2018
Chronological Age: 8 years, 1 month
Gender: Female
Grade: 2nd Grade
Date of Evaluation: February 10 & 17, 2026
Date of Report: April 3, 2026
Evaluator: <PERSON>, PsyD
Licensed Clinical Psychologist
<ORGANIZATION>
Referring Provider: <PERSON>, MD (Pediatrics)
REASON FOR REFERRAL
<PERSON> was referred for a comprehensive psychological evaluation due to concerns from parents and teachers regarding difficulties with attention, emotional regulation, social interactions, and academic engagement. Symptoms have been present for approximately two years and are impacting school performance and family functioning. Parents requested diagnostic clarification and recommendations for support.
BACKGROUND INFORMATION
Developmental History
Pregnancy, delivery, and early developmental milestones were reported as typical and within normal limits. <PERSON> sat unsupported at 7 months, walked at 13 months, used single words by 12 months, and began combining words by 20 months. Sleep patterns are irregular with occasional bedtime resistance. Appetite is generally good.
Medical History
No major illnesses, injuries, or hospitalizations. Routine vision and hearing screenings are within normal limits. No current medications. No history of seizures, significant head trauma, or toxic exposures.
Family History
<PERSON> lives with both biological parents who are married. There is a reported family history of attention and anxiety-related difficulties on the paternal side. No other significant psychiatric or developmental disorders noted in immediate family. The family environment is described as supportive with consistent routines.
Educational History
<PERSON> attends a general education 2nd-grade classroom in a public school. Teachers report challenges with sustaining attention during group and independent work, difficulty completing assignments on time, and occasional emotional outbursts when frustrated. Academic skills appear to be in the average range when structure is provided, though progress is inconsistent due to inattention and disorganization.
Social/Behavioral History
<PERSON> has a small group of friends but often has trouble sharing or managing disappointment during play. At home, she can be argumentative about daily routines and homework. No history of aggression toward others or destruction of property. Reported strengths include creativity, kindness toward younger children, and strong expressive language skills.
ASSESSMENT PROCEDURES

Clinical interview with parents (approximately 2 hours)
Clinical interview with <PERSON> (40 minutes)
Teacher telephone interview and standardized rating scales
Wechsler Intelligence Scale for Children, Fifth Edition (WISC-V)
Conners 4th Edition – Parent and Teacher Forms
Behavior Assessment System for Children, Third Edition (BASC-3) – Parent, Teacher, and Self-Report
Test of Variables of Attention (TOVA)
Children’s Depression Inventory, Second Edition (CDI-2)
Screen for Child Anxiety Related Emotional Disorders (SCARED) – Parent and Child versions
Review of available school and pediatric records

BEHAVIORAL OBSERVATIONS
<PERSON> arrived on time and was initially shy but warmed up quickly. She was talkative and cooperative yet displayed frequent fidgeting, shifting in her seat, and occasional off-task behaviors. Attention to tasks varied; she required moderate redirection. Speech was clear and fluent. Affect was generally euthymic with some anxiety noted during challenging tasks. Effort and motivation appeared adequate throughout the evaluation sessions.
TEST RESULTS AND CLINICAL FINDINGS
Cognitive Ability
On the WISC-V, <PERSON>’s overall intellectual functioning fell in the Average to High Average range (Full Scale IQ = 112, 79th percentile). Strongest performance was observed in Verbal Comprehension and Fluid Reasoning. Processing Speed and Working Memory indices were lower relative to other domains, falling in the low average range and consistent with reported attentional concerns.
Attention and Executive Functioning
Results from the Conners 4 and TOVA indicated clinically significant difficulties with inattention and impulsivity in both home and school environments (T-scores ranging from 68–76). Performance on the TOVA showed elevated omission and commission errors along with high variability in response time, supporting deficits in sustained attention and inhibitory control.
Social-Emotional and Behavioral Functioning
BASC-3 profiles revealed clinically significant elevations in hyperactivity, attention problems, and emotional self-regulation across reporters. SCARED scores indicated elevated anxiety symptoms (parent total score = 28, above clinical cutoff), particularly related to separation and school performance. CDI-2 results did not suggest significant depressive symptoms. Teacher ratings were somewhat higher than parent ratings, indicating greater impairment in the structured school setting.
Diagnostic Impression
Based on a review of developmental history, behavioral observations, multi-informant standardized measures, and cognitive testing, <PERSON> meets DSM-5-TR criteria for the following diagnoses:
1. Attention-Deficit/Hyperactivity Disorder, Combined Presentation (F90.2)
Moderate severity. Symptoms are present across multiple settings and cause clinically significant impairment in academic and social functioning.
2. Separation Anxiety Disorder (F93.0)
Mild severity. Excessive fear of separation from attachment figures that interferes with daily activities.
No evidence of a specific learning disorder, autism spectrum disorder, or oppositional defiant disorder was identified at this time. Notable strengths include solid verbal abilities and a supportive family system.
RECOMMENDATIONS

Medical Consultation – Referral to a child and adolescent psychiatrist for consideration of medication management options for ADHD symptoms.
Behavioral Interventions – Participation in evidence-based parent training program (e.g., 8–12 sessions of behavioral parent management training) to address compliance and emotional regulation at home.
School Supports – Request a Section 504 Plan or IEP meeting to implement classroom accommodations such as preferential seating, extended time on assignments/tests, frequent breaks, and use of organizational tools. Daily behavior report card recommended.
Individual Therapy – Weekly cognitive-behavioral therapy (CBT) focused on anxiety management, coping skills, and building emotional regulation. Services can be coordinated through <ORGANIZATION> or community providers.
Follow-up – Re-evaluation in 12 months or earlier if significant changes occur. Ongoing monitoring of academic progress and response to interventions is advised.

Summary Statement
<PERSON> is a bright and engaging 8-year-old girl presenting with ADHD, Combined Presentation and co-occurring Separation Anxiety Disorder. With timely, evidence-based intervention including behavioral support, school accommodations, and therapy, significant improvement in attention, anxiety, and overall functioning is anticipated.
Signed:
<PERSON>, PsyD
Licensed Psychologist
<ORGANIZATION>
Report reviewed by:
<PERSON>, MD
Medical Director, <ORGANIZATION>
    """
)

doc2 = DocumentSource(
    id="doc_002",
    text_content="""
    <ORGANIZATION> CLINICAL EVALUATION REPORT

Date of Evaluation: October 14, 2025
Date of Report: October 28, 2025
Client Name: <PERSON>
Date of Birth: May 12, 2017 (Age: 8 years, 5 months)
Grade: 3rd Grade
Clinician: <PERSON>, Psy.D.

REASON FOR REFERRAL
<PERSON> was referred to <ORGANIZATION> by their parents for a comprehensive diagnostic evaluation. Concerns include persistent difficulties with sustained attention, frequent "daydreaming" in class, and significant emotional reactivity when transitioning between preferred and non-preferred tasks. Their teacher at <ORGANIZATION> School noted that while <PERSON> is academically capable, they often fail to complete independent work and struggle with social cues during recess.

ASSESSMENT PROCEDURES
The following assessment tools and data sources were utilized:

Clinical Interview: With parents and <PERSON>.

Conners 4: Parent and Teacher Rating Scales.

Behavior Assessment System for Children (BASC-3): Parent and Teacher Forms.

Wechsler Intelligence Scale for Children (WISC-V): Selected subtests.

Review of Records: Previous report cards and pediatrician notes.

CLINICAL OBSERVATIONS
<PERSON> presented as a cooperative and engaging child. They established age-appropriate eye contact but frequently shifted in their seat, occasionally standing up to look at items on the clinician's shelf. When tasks became challenging, <PERSON> exhibited "avoidant" behaviors, such as asking for water breaks or changing the subject. They showed a tendency toward "all-or-nothing" thinking (e.g., "I’m bad at math because I missed this one").

TEST RESULTS & DATA INTEGRATION
Cognitive and Academic Functioning
Preliminary results from the WISC-V suggest that <PERSON> possesses High Average verbal reasoning skills. However, their Working Memory and Processing Speed scores fell within the Low Average range. This "bottleneck" effect explains why <PERSON> understands complex concepts but struggles to get them down on paper.

Behavioral and Emotional Functioning
Data from the BASC-3 and Conners 4 indicate clinically significant elevations in the following areas:

Inattention: Both parents and teachers report that <PERSON> loses track of instructions and requires frequent redirection.

Emotional Reactivity: Scores suggest <PERSON> experiences "big feelings" more intensely than peers, often resulting in tearfulness or shut-down behaviors when frustrated.

Executive Functioning: Deficits were noted in planning, organizing, and initiating tasks.

DIAGNOSTIC IMPRESSION
Based on the current evaluation, <PERSON> meets the diagnostic criteria for the following:

Attention-Deficit/Hyperactivity Disorder, Combined Presentation (F90.2): Evidenced by a persistent pattern of inattention and hyperactivity that interferes with functioning in two or more settings.

Generalized Anxiety Disorder (F41.1): Characterized by excessive worry regarding academic performance and social rejection, contributing to their emotional reactivity.

SUMMARY AND RECOMMENDATIONS
<PERSON> is a bright and creative child whose primary challenges stem from executive functioning deficits and anxiety. To support their development, the following is recommended:

School-Based Supports
Individualized Education Program (IEP) or 504 Plan: To provide accommodations such as extended time on tests and preferential seating away from distractions.

Task Breakdown: Assignments should be broken into smaller, manageable "chunks" with frequent check-ins.

Visual Schedules: Utilize a visual timer or checklist to help <PERSON> navigate transitions.

Therapeutic Interventions
Cognitive Behavioral Therapy (CBT): To help <PERSON> identify and challenge "anxious thoughts" and develop coping strategies for frustration.

Parent Management Training (PMT): To assist parents in implementing consistent behavioral structures and positive reinforcement at home.

Medical Consultation
It is recommended that the family share this report with <PERSON>’s pediatrician or a child psychiatrist to discuss the potential role of pharmacological support for ADHD symptoms.

Signature:
<PERSON>, Psy.D.
Clinical Psychologist, <ORGANIZATION>
    """
)



doc3 = DocumentSource(
    id="doc_003",
    text_content="""
    <ORGANIZATION> CLINICAL EVALUATION REPORT

Date of Evaluation: November 05, 2025
Date of Report: November 19, 2025
Client Name: <PERSON>
Date of Birth: August 22, 2016 (Age: 9 years, 2 months)
Grade: 4th Grade
Clinician: <PERSON>, Ph.D.

REASON FOR REFERRAL
<PERSON> was referred to <ORGANIZATION> for a diagnostic evaluation due to increasing academic friction and interpersonal difficulties. At home, parents report significant "meltdowns" during the evening routine. At school, <PERSON>’s teacher at <ORGANIZATION> Academy describes them as a "bright but inconsistent" student who frequently disrupts the flow of the classroom with off-topic comments or refusal to comply with group instructions.

BEHAVIORAL OBSERVATIONS AND EXAMPLES
Home Environment
Parental interviews and home-based rating scales reveal a pattern of executive functioning deficits that manifest in daily stressors:

The "Morning Logjam": <PERSON> is unable to complete a three-step routine (brush teeth, get dressed, put on shoes) without a minimum of four to five verbal prompts. They are frequently found distracted by a toy or staring into space halfway through dressing.

Task Avoidance: Homework sessions often last three hours for assignments that should take twenty minutes. <PERSON> exhibits "escape behaviors," such as complaining of physical ailments (e.g., stomach aches) or dropping pencils to stall the initiation of writing tasks.

Emotional Dysregulation: Small triggers, such as being told a favorite snack is unavailable, result in "Level 10" emotional responses (screaming or slamming doors) that can last up to thirty minutes.

School Environment
Teacher reports and classroom observations highlight challenges with self-regulation and social pragmatics:

Impulse Control: <PERSON> frequently blurt out answers before the teacher has finished the question. During "carpet time," they struggle to keep their hands to themselves, often poking peers or playing with their shoelaces.

Executive Dysfunction: Their desk is described as "chronically disorganized," with crumpled papers and missing supplies. <PERSON> often forgets to turn in completed work, even when it is tucked inside their backpack.

Social Reciprocity: During recess, <PERSON> struggles to negotiate the rules of a game. If a peer suggests a change to the rules, <PERSON> may become rigid, declare the game "stupid," and walk away, leading to social isolation.

TEST RESULTS & CLINICAL DATA
Cognitive Testing (WISC-V)
<PERSON> demonstrated Superior range performance in Fluid Reasoning, indicating a high capacity for solving novel, abstract problems. However, their Processing Speed fell into the Borderline range.

Clinical Significance: In the classroom, <PERSON> understands the material faster than their peers but cannot output the work at the same pace. This leads to a "performance gap" that fuels their frustration and subsequent behavioral outbursts.

Rating Scales (BASC-3 & BRIEF-2)
Inhibition: Clinically significant. <PERSON> struggles to "stop and think" before acting.

Shift: Elevated. <PERSON> has extreme difficulty moving from a high-interest activity (e.g., Video Games/Recess) to a low-interest activity (e.g., Math/Chores).

DIAGNOSTIC IMPRESSION
Based on the integrated data, <PERSON> meets criteria for:

Attention-Deficit/Hyperactivity Disorder, Combined Presentation (F90.2): Primary drivers are poor impulse control and significant distractibility.

Oppositional Defiant Disorder (F91.3): Characterized by a frequent pattern of irritability and defiance toward authority figures, largely triggered by transitions and cognitive fatigue.

COMPREHENSIVE RECOMMENDATIONS
Targeted Educational Strategies
The "First/Then" Principle: Use visual boards to show "First: 5 Math Problems, Then: 2 Minutes of Drawing." This reduces the perceived "infinity" of the non-preferred task.

Heavy Work Breaks: Allow <PERSON> to perform "heavy work" (e.g., carrying the mail to the office or doing wall pushes) to provide proprioceptive input when they appear hyperactive.

Home-Based Interventions
Collaborative & Proactive Solutions (CPS): Shift away from a "punishment" model to a "problem-solving" model. Identify the "lagging skills" that cause the meltdowns and work with <PERSON> to find solutions during calm periods.

Environment Design: Minimize visual clutter in <PERSON>'s study area. Use a "launchpad" by the front door—a specific bin for their backpack and shoes—to reduce morning friction.

Therapeutic Plan
Social Skills Group: To practice "perspective-taking" and flexible thinking in a supervised peer setting.

Child-Parent Psychotherapy: To repair the parent-child bond which has been strained by chronic behavioral conflict.

Signature:
<PERSON>, Ph.D.
Clinical Director, <ORGANIZATION>

    """
)

    
docs = [doc1, doc2, doc3]


## 2. Initialize the Orchestrator with Tracking Enabled
We configure the `Orchestrator` with our generalization metrics enabled, utilizing `save_stage_outputs=True` to persist checkpoints tracking intermediate vectors.


In [3]:
orchestrator = Orchestrator(
    max_concurrency=4,
    hallucination_filter=True,
    ontology_depth=None,
    strict_typing=True,
    verbose=True,
    generalize_latent_space=True,
    generalize_structural_roles=True,
    generalize_taxonomic_lifting=True,
    generalize_seeded_schemas=True,
    save_stage_outputs=True
)


[*] Orchestrator initialized. Model: mistral-nemo:12b-instruct-2407-q4_K_M | Base URL: http://localhost:11434/v1


## 3. Phase 1: Discovery & Abstraction
Instead of running `run_pipeline`, we manually invoke `run_discovery` to extract raw topologies, cluster them, and synthesize a programmatic output schema.
Because `save_stage_outputs` is enabled, this will export `1_raw_triples.json`, `2_hardened_clusters.json`, and `3_discovered_schema.json`.


In [4]:
# NOTE: This runs the extraction natively over the LLM. 
# If you are doing downstream testing, you can skip this and simply load "stage_outputs/3_discovered_schema.json"
discovered_schema = await orchestrator.run_discovery(docs)

print(f"\nSuccessfully Dynamically Instantiated BaseModel: {discovered_schema.__name__}")


====== STAGE 0. DISCOVERY LOOP ======
[*] Executing schema-less triple extraction over 3 documents...
[TIMER] ExplorerEngine.extract_raw_triples for doc 'doc_002' took: 25.31s
[CHECKPOINT] Document 'doc_002' done scanning for triples.
[TIMER] ExplorerEngine.extract_raw_triples for doc 'doc_003' took: 58.06s
[CHECKPOINT] Document 'doc_003' done scanning for triples.
[TIMER] ExplorerEngine.extract_raw_triples for doc 'doc_001' took: 61.48s
[CHECKPOINT] Document 'doc_001' done scanning for triples.
[+] Extracted 44 raw topological triples.
[SAVE] Stage output written to stage_outputs/1_raw_triples.json
[*] Running algorithmic Louvain community detection natively routing the graph...
[VERBOSE] K-Core Pruning: Sequestered 59 leaf noise nodes. Core graph contains 4 dense nodes.
[VERBOSE] Auto-Resolution Sweeper: Target max bounding limit = 6 communities.
[VERBOSE] Auto-Resolution stabilized at sensitivity scale 1.50 rendering 4 core distinct logic atoms.
[VERBOSE] Structural Jaccard Merger: 

## 4. Validating Downstream Capabilities
Let's simulate a downstream process testing our logic by loading the exported `2_hardened_clusters.json` file.


In [5]:
# We can load these datasets computationally without querying the LLM again.
try:
    with open("stage_outputs/2_hardened_clusters.json", "r") as f:
        hardened_clusters = json.load(f)
        print(f"Loaded {len(hardened_clusters)} Structural Abstractions natively from Local Disk for isolated testing:")
        print(json.dumps(hardened_clusters[0], indent=2))
except Exception as e:
    print(f"Failed to load isolated stage output: {e}")


Loaded 4 Structural Abstractions natively from Local Disk for isolated testing:
{
  "reasoning": "[Seeded Meta] The class 'Intermediate-Bridge-1' represents timepoints or actions via its predicate relationships ('DateOfEvaluation -> DateTime', 'DateOfReport -> DateTime'), making it an 'Action' in the Golden Schema.",
  "class_name": "Action",
  "nodes": [
    "<DATE>",
    "October 14, 2025",
    "October 28, 2025"
  ],
  "canonical_predicates": [
    "DateOfEvaluation -> DateTime",
    "DateOfReport -> DateTime"
  ],
  "negative_constraints": [
    "NEVER - (Not a TemporalEvent: HasLocation)",
    "NEVER - (Not a TemporalEvent: HasPhysicalAddress)"
  ]
}


## 5. Phase 2: Information Distillation (Constrained Extraction)
Now we force the original unstructured context into the constraints of the dynamic schema via validation loop.
This will save its exact final models to `4_final_distilled_outputs.json`.


In [6]:
# Extract empirical features against the dynamically discovered schema
constrained_outputs = []
sem = asyncio.Semaphore(orchestrator.max_concurrency)
tasks = [orchestrator._safe_extract_features(doc, discovered_schema, sem) for doc in docs]
constrained_outputs = await asyncio.gather(*tasks)

print("\nSuccess! Context maps to Generalized Topologies natively.")


[TIMER] Constrained Extraction for doc 'doc_002' took: 25.88s
[CHECKPOINT] Document 'doc_002' done constrained mapping.

[VERBOSE] Constrained Payload:
{
  "action": {
    "dateofevaluation____datetime": "October 14, 2025",
    "dateofreport____datetime": "October 28, 2025"
  },
  "agent": {
    "name": "<PERSON>",
    "date_of_birth": "May 12, 2017",
    "grade": "3rd Grade",
    "clinician": "<PERSON>",
    "referral_reason": "Persistent difficulties with sustained attention, frequent daydreaming in class, emotional reactivity when transitioning between preferred and non-preferred tasks, struggling to complete independent work, social cues during recess",
    "cooperative_status": "Cooperative and engaging child",
    "eye_contact": "Established age-appropriate eye contact but frequently shifted in their seat",
    "seat_behavior": "Occasionally standing up to look at items on the clinician's shelf",
    "task_avoidance": "'Avoidant' behaviors, such as asking for water breaks or chan

## 6. Inspect Final Dataset
Alternatively, you can skip full runtime computation and build entire frontend charts directly utilizing `stage_outputs/4_final_distilled_outputs.json`


In [7]:
for i, item in enumerate(constrained_outputs):
    print(f"--- Processed Document ID: {docs[i].id} ---")
    print(item.model_dump_json(indent=2))
    print("\n")


--- Processed Document ID: doc_001 ---
{
  "action": {
    "dateofevaluation____datetime": "February 10 & 17, 2026",
    "dateofreport____datetime": "April 3, 2026"
  },
  "agent": {
    "name": "<PERSON>",
    "date_of_birth": "March 12, 2018",
    "grade": "2nd Grade",
    "clinician": "<ORGANIZATION>",
    "referral_reason": "School performance and family functioning",
    "cooperative_status": "cooperative",
    "eye_contact": "eye contact maintained",
    "seat_behavior": "fidgeting, shifting in seat, occasional off-task behaviors",
    "task_avoidance": "attention to tasks varied; she required moderate redirection",
    "thinking_style": "talkative",
    "inattention__parental_concerns_": "Inattention and impulsivity in home and school environments (T-scores ranging from 68–76).",
    "emotional_reactivity__score_": "elevated anxiety symptoms (parent total score = 28, above clinical cutoff)",
    "executive_functioning": "deficits in sustained attention and inhibitory control.",
